In [1]:
import pandas as pd
import numpy
import json
import os
os.chdir('..')

In [ ]:
from copy import deepcopy

In [2]:
orders = ['aos', 'aso', 'sao', 'oas', 'osa']

In [4]:
dataset_type = 'hoasa_hotel'
lang = 'indo'
ori_data_path = f'dataset/{dataset_type}/{lang}/mvp_aos/train.json'
with open(ori_data_path, 'r') as f:
    ori_data = json.load(f)

In [9]:
from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
	"""
	Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
	Each dictionary contains the tag as the key and the corresponding value.
	For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
	[{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
	{'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

	Args:
		text (str): ABSA string output to be parsed.

	Returns:
		List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

	"""
	pattern = r"\[(\w+)\]\s*([^[]+)"
	matches = re.findall(pattern, text)

	result = []
	current_dict = {}

	for tag, content in matches:
		if tag == "SSEP":  # Sentence separator -> Start a new dictionary
			result.append(current_dict)
			current_dict = {}
		else:
			current_dict[tag] = content.strip()

	if current_dict:  # Append the last sentence if it exists
		result.append(current_dict)

	return result

def convert_output_to_mvp_format(data_list: List[Dict[str, str]], order='aos') -> str:
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the MVP paper.

	Args:
		data_list (List[Dict[str, str]]): A list of strings, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).
		order (str): A string specifying the order of elements (e.g., 'aos', 'ao', 'as', 'a', 'o')

	Returns:
		A single string formatted with special tokens and elements based on the order
	"""

	result_str = []
	for triplet in data_list:
		triplet_str = [f"[{element.upper()}] {triplet[element.upper()]}" for element in order]
		triplet_str = ' '.join(triplet_str)
		result_str.append(triplet_str)
	return ' [SSEP] '.join(result_str)

def convert_input_to_mvp_format(input_str: str, order='aos') -> str:
	order_str = ' '.join([f"[{element.upper()}]" for element in order])
	final_str = input_str.replace('[A] [O] [S]', order_str)
	return final_str

In [12]:
print(convert_output_to_mvp_format(parse_absa_string('[A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive'), order='aso'))
print(convert_input_to_mvp_format('test 1 2 3 [A] [O] [S]', order='ao'))

[A] harga [S] positive [O] terjangkau [SSEP] [A] fasilitas [S] positive [O] nyaman
test 1 2 3 [A] [O]


In [14]:
ori_data[0]

{'sentence_id': 0,
 'instance_id': 0,
 'input': 'kamar saya ada kendala di ac tidak berfungsi optimal . dan juga wifi koneksi kurang stabil . [A] [O] [S]',
 'target': '[A] ac [O] tidak berfungsi optimal [S] negative [SSEP] [A] wifi koneksi [O] kurang stabil [S] negative',
 'element_order': 'aos',
 'task_elements': 'aos',
 'dataset_type': 'hotel_reviews'}

In [17]:
new_data = []
instance_id = 0
for instance in ori_data:
	for order in orders:
		new_data.append({
			'sentence_id': instance['sentence_id'],
			'instance_id': instance_id,
			'input': convert_input_to_mvp_format(instance['input'], order),
			'target': convert_output_to_mvp_format(parse_absa_string(instance['target']), order),
			'element_order': order,
			'task_elements': 'aos',
			'dataset_type': instance['dataset_type']
		})
		instance_id += 1
assert len(new_data) == len(ori_data) * len(orders)

In [24]:
test = [1,2,3]
test[5:9]

[]

In [25]:
new_path = f'dataset/{dataset_type}/{lang}/mvp/train.json'
os.makedirs(os.path.dirname(new_path), exist_ok=True)
with open(new_path, 'w') as f:
    json.dump(new_data, f, indent=4, ensure_ascii=False)

In [ ]:
# [
#     {
#         "sentence_id": 3500,
#         "instance_id": 17500,
#         "task_elements": "aos",
#         "input": "pelayanan nya sangat ramah . [A] [O] [S]",
#         "target": "[A] pelayanan nya [O] sangat ramah [S] positive",
#         "element_order": "aos"
#     },
#     {
#         "sentence_id": 3500,
#         "instance_id": 17501,
#         "task_elements": "aos",
#         "input": "pelayanan nya sangat ramah . [A] [S] [O]",
#         "target": "[A] pelayanan nya [S] positive [O] sangat ramah",
#         "element_order": "aso"
#     },
#     {
#         "sentence_id": 3500,
#         "instance_id": 17502,
#         "task_elements": "aos",
#         "input": "pelayanan nya sangat ramah . [S] [A] [O]",
#         "target": "[S] positive [A] pelayanan nya [O] sangat ramah",
#         "element_order": "sao"
#     },
#     {
#         "sentence_id": 3500,
#         "instance_id": 17503,
#         "task_elements": "aos",
#         "input": "pelayanan nya sangat ramah . [O] [A] [S]",
#         "target": "[O] sangat ramah [A] pelayanan nya [S] positive",
#         "element_order": "oas"
#     },
#     {
#         "sentence_id": 3500,
#         "instance_id": 17504,
#         "task_elements": "aos",
#         "input": "pelayanan nya sangat ramah . [O] [S] [A]",
#         "target": "[O] sangat ramah [S] positive [A] pelayanan nya",
#         "element_order": "osa"
#     },